# APIs y HTTP

Este notebook es la clase de la sección 10.1 completa: cada celda trae su explicación arriba y el código listo. **Antes de empezar, arranca el servidor** en una terminal aparte, dentro de esta carpeta (`uv run python servidor.py`), y déjala visible: ahí aparece cada petición que hagas.

Córrelas **en orden**, una por una (Shift+Enter), y compara con «Deberías ver». La página del curso tiene lo mismo, más las tablas y los diagramas: https://rayalucaria.org/fdd_o26/apis/apis-y-http/

# APIs y HTTP

### Arranque del notebook

**Haz (celda 0.1):** importa lo de la clase y comprueba que el servidor contesta.

## 0.1 · Arranque: ¿contesta el servidor?

In [ ]:
import re
import time

import requests
from cable import TOKEN, URL, a_mano, total, ver

r = requests.get(f"{URL}/ventas", timeout=5)
print(r.status_code, "· ventas guardadas:", total())

**Qué hace cada línea:**

- `import requests` — la librería que manda peticiones HTTP desde Python.
- `from cable import …` — trae de `cable.py` la dirección del servidor (`URL`), el token para escribir (`TOKEN`) y las ayudas `a_mano`, `ver` y `total`.
- `requests.get(f"{URL}/ventas", timeout=5)` — le pide al servidor la lista de ventas; si en 5 segundos no contesta, se rinde.
- `total()` — cuenta las ventas que el servidor tiene guardadas.

**Deberías ver:**

```text
200 · ventas guardadas: 4
```

- Si sale `ConnectionError` (*connection refused*), el servidor no está corriendo: vuelve al Haz 3.
- En la terminal del servidor aparecieron dos peticiones: la tuya y la de `total()`.

# Qué es una API

Meta: saber qué es una API, en qué se distingue una API de biblioteca de una API web, y qué cambia cuando una llamada cruza la red.

Meta: saber qué es una API, en qué se distingue una API de biblioteca de una API web, y qué cambia cuando una llamada cruza la red.

| Término | Qué es | Lo ves con |
|---|---|---|
| **API** | Las operaciones que un programa ofrece a otros, con las reglas para usarlas | la celda 1.1 |
| **contrato** | Lo que la API promete: qué entra, qué sale, qué errores puede dar | el JSON de la celda 1.1 |
| **implementación** | El código que cumple el contrato; quien usa la API no lo ve | `servidor.py` |
| **cliente** | El programa que pide | tu notebook |
| **servidor** | El programa que recibe la petición y contesta | la terminal del servidor |
| **endpoint** | Una dirección concreta de una API web donde se pide una operación | `GET /ventas` |
| **latencia** | El tiempo entre que pides algo y te llega la respuesta | la celda 1.2 |
| **JSON** | Un formato de texto para datos: objetos `{…}`, listas `[…]`, números, texto, `true`/`false` y `null` | la respuesta de la celda 1.1 |

### En corto

- **Una API es un contrato**: dice *qué* puedes pedir y *qué* te regresa, nunca *cómo* lo calcula.
- **Una API web agrega red y serialización a cada llamada**: en una operación pequeña ese costo domina (miles de veces más lenta que una función, en la celda 1.2), y aparecen fallas que una función no tiene.
- **El contrato lo controla otro**: puede cambiarlo, cobrarlo o apagarlo.

### Una API es un contrato

**Definición.** Una **API** (*Application Programming Interface*, interfaz de programación de aplicaciones) es el conjunto de operaciones que un programa ofrece a otros programas, junto con las reglas para usarlas.

Las reglas forman el **contrato**. Un contrato completo dice cinco cosas:

| Parte | Pregunta que contesta | En `len()` de Python |
|---|---|---|
| **Operación** | ¿Qué puedo pedir? | el largo de algo |
| **Entrada** | ¿Qué datos mando y en qué forma? | un objeto con largo: lista, texto, dict |
| **Salida** | ¿Qué me regresa? | un entero ≥ 0 |
| **Errores** | ¿Qué pasa si pido mal? | `TypeError` si el objeto no tiene largo |
| **Reglas de uso** | ¿Límites, permisos, costo? | ninguna |

**Para qué existe.** Separa **interfaz** (el contrato) de **implementación** (el código que lo cumple). Quien usa la API depende sólo del contrato; quien la mantiene puede cambiar la implementación —otro algoritmo, otra base de datos, otro lenguaje— sin romper a nadie, mientras el contrato siga igual.

**Si lo ignoras.** Si tu código depende de un detalle que el contrato no promete (el orden en que llegan los resultados, un campo que «siempre» viene), se rompe el día que cambia la implementación, aunque nadie haya roto el contrato.

**Haz (celda 1.1):** pídele al servidor la lista de operaciones que ofrece.

## 1.1 · El contrato del servidor

In [ ]:
r = requests.get(f"{URL}/")
print(r.status_code)
for ruta, que_hace in r.json()["endpoints"].items():
    print(f"{ruta:22} {que_hace}")

**Qué hace cada línea:**

- `requests.get(f"{URL}/")` — pide la raíz del servidor, `/`, donde este servidor publica la lista de lo que ofrece.
- `r.status_code` — el número con el que el servidor dice cómo le fue; `200` es «bien». La página 3 los explica todos.
- `r.json()["endpoints"]` — convierte la respuesta, que llegó como texto JSON, en un dict de Python, y toma la parte `endpoints`.
- `f"{ruta:22} …"` — imprime cada endpoint alineado a 22 caracteres.

**Deberías ver:**

```text
200
GET /ventas            lista las ventas (filtra con ?tienda=Norte)
GET /ventas/{id}       una venta
POST /ventas           crea una venta (pide token)
PUT /ventas/{id}       reemplaza una venta completa (pide token)
DELETE /ventas/{id}    borra una venta (pide token)
GET /exportar          las ventas en CSV, desde un endpoint viejo
GET /panel             una página HTML con enlaces
GET /borrar?id={id}    borra con GET (mal diseño, a propósito)
POST /reiniciar        regresa los datos al inicio
```

- **Esto es una parte del contrato**: las operaciones. Un contrato completo también fija entradas, salidas, errores y reglas; las páginas 2 y 3 los muestran en acción. Nada de esto dice cómo guarda las ventas: eso es implementación.
- Las APIs reales publican su contrato en una página de documentación, y muchas también en un archivo que las máquinas leen (el formato más común se llama **OpenAPI**).

### API de biblioteca y API web

Hay dos tipos de API según **dónde corre** el código que la cumple.

| | API de biblioteca | API web |
|---|---|---|
| **Dónde corre** | En tu mismo proceso | En otro programa, casi siempre en otra máquina |
| **Cómo la llamas** | Llamas una función: `len(x)` | Mandas un mensaje por la red: `GET /ventas` |
| **Cómo viajan los datos** | Como objetos de Python, en memoria | Como bytes: hay que **serializarlos** (convertirlos a texto o bytes, p. ej. JSON) y deserializarlos |
| **Cuánto tarda** | Nanosegundos | Milisegundos o segundos |
| **Cómo falla** | Un error, al instante | Error, silencio, o respuesta que nunca llega |
| **Quién la cambia** | Tú decides cuándo actualizas la librería | El dueño del servidor, cuando él quiera |

```text
  API de biblioteca                    API web

  ┌─ tu proceso ───────────┐         ┌─ tu proceso ─┐                ┌─ otro proceso ─┐
  │  tu código             │         │  tu código   │   petición     │  servidor      │
  │     │ len(x)           │         │  (cliente)   │ ─────────────▶ │                │
  │     ▼                  │         │              │   respuesta    │                │
  │  código de Python      │         │              │ ◀───────────── │                │
  └────────────────────────┘         └──────────────┘      red       └────────────────┘
```

**Cliente** y **servidor** son *papeles*, no máquinas: el cliente es quien pide y el servidor quien contesta. Tu laptop es cliente cuando le pides datos a una API, y servidor cuando corres `servidor.py`. Hoy es las dos cosas.

Un **endpoint** es la combinación de método y ruta donde la API web ofrece una operación: `GET /ventas` y `POST /ventas` son dos endpoints distintos sobre la misma ruta.

### Lo que cambia al cruzar la red

**Haz (celda 1.2):** la misma consulta, como función y como llamada a la API.

## 1.2 · Función contra red

In [ ]:
VENTAS = {1: {"id": 1, "producto": "café", "precio": 45.0}}

def venta(i):
    return VENTAS[i]

inicio = time.perf_counter()
for _ in range(1000):
    venta(1)
funcion = (time.perf_counter() - inicio) / 1000

sesion = requests.Session()
inicio = time.perf_counter()
for _ in range(200):
    sesion.get(f"{URL}/ventas/1")
red = (time.perf_counter() - inicio) / 200

print(f"función: {funcion * 1e6:10.3f} µs")
print(f"red:     {red * 1e6:10.3f} µs")
print(f"la red fue {red / funcion:,.0f} veces más lenta")

**Qué hace cada línea:**

- `def venta(i)` — la consulta como función: busca la venta en un dict en memoria.
- `time.perf_counter()` — un reloj de alta precisión; la resta de dos lecturas da segundos.
- `for _ in range(1000): venta(1)` — llama la función mil veces y luego divide entre mil: el tiempo de una llamada.
- `requests.Session()` — un cliente que reutiliza la misma conexión de red entre peticiones; sin él, cada petición abriría una conexión nueva y tardaría todavía más.
- `sesion.get(f"{URL}/ventas/1")` — la misma consulta, pero al servidor, por la red. 200 veces.
- `* 1e6` — convierte segundos a microsegundos (µs, millonésimas de segundo).

**Deberías ver** (en la máquina del profesor; tus números cambian, la proporción se parece):

```text
función:      0.127 µs
red:       1125.267 µs
la red fue 8,890 veces más lenta
```

- **Para una consulta así de pequeña, miles de veces más lenta, y sin salir de tu máquina.** Si el servidor hace algo pesado (un cálculo de minutos), la red pesa menos en el total; la proporción depende del trabajo y del entorno. El servidor corre en la misma laptop; con un servidor en otro país hay que sumar decenas o cientos de milisegundos de viaje.
- En la terminal del servidor pasaron 200 peticiones. Cada una se **serializó** a texto, viajó, se leyó, se respondió y se volvió a convertir en dict.

**Por qué importa al diseñar.** Un ciclo que llama una API una vez por fila es razonable con 100 filas y una catástrofe con un millón. Por eso las APIs ofrecen pedir muchos datos en una sola petición, y por eso conviene guardar en disco lo que ya descargaste.

### Tres supuestos falsos sobre la red

En 1994, Peter Deutsch, en Sun Microsystems, escribió una lista de supuestos que los programadores hacen sin darse cuenta y que siempre fallan; se conocen como las **falacias de la computación distribuida** (eran siete; James Gosling agregó la octava hacia 1997). Tres bastan para hoy:

| Supuesto falso | Lo que pasa en realidad | Dónde lo ves en esta clase |
|---|---|---|
| **La red es confiable** | Una petición puede perderse, o llegar y que se pierda la respuesta | celdas 3.3 y 3.4 |
| **La latencia es cero** | Cada llamada cuesta milisegundos, no nanosegundos | celda 1.2 |
| **El ancho de banda es infinito** | Bajar 300 KB por cada consulta se nota en el tiempo y en el costo | celda 4.1 |

La segunda fila tiene una consecuencia que la página 3 trabaja a fondo: una **falla parcial**. Cuando no llega la respuesta, el cliente no sabe si el servidor hizo el trabajo o no. En una función eso no existe: o regresó, o lanzó un error.

### El contrato es de otro

Cuando consumes una API web, el contrato y el servidor son de alguien más. Puede cambiarlo, cobrarlo o apagarlo, y tu código no tiene voto.

- **Twitter, febrero de 2023.** Anunció el 2 de febrero que el acceso gratuito a su API terminaba el 9 de febrero, con una semana de aviso; semanas antes ya había cortado a las apps de terceros como Tweetbot.
- **Reddit, junio de 2023.** Empezó a cobrar US$0.24 por cada 1,000 llamadas. El autor de Apollo, una app muy usada para leer Reddit, estimó que eso le costaría unos US$20 millones al año, y la cerró el 30 de junio de 2023.

**Para una fuente de datos esto significa:** guarda la respuesta cruda de cada descarga (el JSON tal como llegó) antes de transformarla. Si mañana la API cambia o desaparece, tus datos de ayer siguen ahí y puedes reprocesarlos.

### ¿API o archivo?

Muchas fuentes ofrecen las dos cosas: una API y un archivo descargable (CSV, Parquet). Elegir es una decisión de diseño.

| Si… | Conviene | Por qué |
|---|---|---|
| Los datos cambian seguido y necesitas lo último | **API** | El archivo envejece en cuanto lo bajas |
| Necesitas una parte pequeña de algo enorme | **API** | Pides sólo esa parte (filtros en la URL) |
| Necesitas todo el histórico, una vez | **Archivo** | Una descarga contra miles de llamadas |
| Vas a repetir el análisis muchas veces | **Archivo** (o guarda lo de la API) | No dependes de que el servidor siga ahí ni pagas cada lectura |
| La API tiene límite de peticiones | **Archivo**, si existe | El límite no aplica a una descarga única |

# El mensaje HTTP

Meta: poder leer —y escribir a mano— una petición y una respuesta HTTP, sabiendo para qué sirve cada parte.

Meta: poder leer —y escribir a mano— una petición y una respuesta HTTP, sabiendo para qué sirve cada parte.

| Término | Qué es | Lo ves con |
|---|---|---|
| **protocolo** | Las reglas que dos programas siguen para entenderse: qué se manda, en qué orden y en qué formato | esta página |
| **TCP** | El protocolo que entrega bytes, completos y en orden, entre dos programas en la red | `a_mano()` |
| **socket** | El objeto con el que un programa abre una conexión TCP y manda o recibe bytes | `cable.py` |
| **HTTP** | El protocolo de la web y de casi todas las APIs: define el formato de petición y respuesta | celda 2.1 |
| **petición** | El mensaje que manda el cliente | celda 2.1 |
| **respuesta** | El mensaje que regresa el servidor | celda 2.1 |
| **header** | Una línea `Nombre: valor` con información *sobre* el mensaje | celda 2.3 |
| **body** | Los datos que lleva el mensaje, si lleva | celda 2.4 |

### En corto

- **Un mensaje HTTP tiene cuatro partes, en este orden**: línea inicial, headers, una línea en blanco y el body (que puede faltar).
- **Los headers describen el mensaje; el body son los datos.** Si un header miente, el body se lee mal.
- **Todo viaja en claro**, salvo que uses HTTPS: un token en `http://` lo lee cualquiera que vea la red.

### Protocolo, TCP y HTTP

**Definición.** Un **protocolo** son las reglas que dos programas siguen para comunicarse: qué mensajes existen, en qué orden se mandan y con qué formato. **HTTP** (*Hypertext Transfer Protocol*) es el protocolo de la web; lo definen documentos públicos de la IETF, el organismo que publica los estándares de internet, llamados **RFC** (el de la semántica de HTTP es el RFC 9110, de 2022).

En esta clase, HTTP/1.1 viaja sobre **TCP**, que entrega los bytes en orden y sin huecos mientras la conexión siga viva; si la conexión se cae, TCP reporta el error, no lo arregla. HTTP decide **qué bytes** mandar; TCP, **cómo llegan**.

```text
  tu código          requests.get(...)            ← tú
  ─────────────────────────────────────────
  HTTP               "GET /ventas HTTP/1.1 ..."   ← el formato del mensaje
  ─────────────────────────────────────────
  TCP                bytes completos y en orden   ← la conexión (un socket)
  ─────────────────────────────────────────
  red                paquetes, wifi, cables
```

Esta página muestra **HTTP/1.1**, donde la línea inicial y los headers son texto que puedes leer. **HTTP/2** manda la misma información en tramas binarias sobre TCP, y **HTTP/3** sobre **QUIC**, otro transporte construido sobre UDP. No tienen una línea inicial de texto, pero el significado de métodos, headers, body y códigos de estado es el mismo: lo que aprendas aquí vale para las tres.

### Las cuatro partes de un mensaje

Petición y respuesta tienen la misma forma:

| Parte | Qué es | Para qué sirve |
|---|---|---|
| **Línea inicial** | La primera línea | En la petición dice qué quieres; en la respuesta, cómo le fue |
| **Headers** | Cero o más líneas `Nombre: valor` | Describen el mensaje: formato, tamaño, quién eres, qué aceptas |
| **Línea en blanco** | Una línea vacía | Marca dónde terminan los headers y empieza el body |
| **Body** | Bytes de datos, opcional | Lo que mandas o recibes: un JSON, un CSV, una imagen |

Cada línea termina con dos caracteres, `\r\n` (retorno de carro y salto de línea), no sólo con `\n`. Así lo exige el estándar.

```text
  PETICIÓN                                    RESPUESTA

  POST /ventas HTTP/1.1          ← línea      HTTP/1.1 201 Created          ← línea
  Host: 127.0.0.1:8026           ┐            Location: /ventas/5           ┐
  Content-Type: application/json │ headers    Content-Type: application/json│ headers
  Content-Length: 75             │            Content-Length: 80            │
  Authorization: Bearer fdd-2026 ┘                                          ┘
                                 ← en blanco                                ← en blanco
  {"tienda": "Norte", ...}       ← body       {"id": 5, "tienda": ...}      ← body
```

**Haz (celda 2.1):** escribe una petición HTTP a mano y mándala por un socket.

## 2.1 · HTTP escrito a mano

In [ ]:
peticion = (
    "GET /ventas/1 HTTP/1.1\r\n"
    "Host: 127.0.0.1:8026\r\n"
    "Connection: close\r\n"
    "\r\n"
)
print(a_mano("127.0.0.1", 8026, peticion))

**Qué hace cada línea:**

- `"GET /ventas/1 HTTP/1.1\r\n"` — la línea de petición: quiero **leer** (`GET`) la venta 1, y hablo HTTP versión 1.1.
- `"Host: 127.0.0.1:8026\r\n"` — a qué servidor va dirigida. Es el único header obligatorio en HTTP/1.1.
- `"Connection: close\r\n"` — pide al servidor que cierre la conexión al terminar; así `a_mano` sabe cuándo dejar de leer.
- `"\r\n"` — la línea en blanco: aquí terminan los headers. Un `GET` no lleva body.
- `a_mano(...)` — abre un socket TCP al puerto 8026, manda el texto tal cual y regresa lo que llegue.

**Deberías ver** (la fecha y el ETag cambian):

```text
HTTP/1.1 200 OK
Server: VentasFDD/1.0
Date: Tue, 20 Oct 2026 22:52:05 GMT
ETag: "d788c34b28aa"
Vary: Accept
Content-Type: application/json
Content-Length: 81

{"id": 1, "tienda": "Centro", "producto": "café", "cantidad": 3, "precio": 45.0}
```

- **No hubo librería de por medio**: escribiste texto, el servidor lo entendió y contestó texto. Eso es todo HTTP/1.1.
- La respuesta tiene las cuatro partes: línea de estado, seis headers, una línea en blanco y el body en JSON.
- En la terminal del servidor aparece tu petición con sólo dos headers: los que escribiste.

**Haz (celda 2.2):** lo mismo, contra un servidor real de internet.

## 2.2 · HTTP a mano contra example.com

In [ ]:
print(a_mano("example.com", 80,
             "HEAD / HTTP/1.1\r\nHost: example.com\r\nConnection: close\r\n\r\n"))

**Qué hace cada línea:**

- `"example.com", 80` — el sitio de ejemplo oficial de internet, en el puerto 80, el de HTTP sin cifrar.
- `HEAD /` — el método `HEAD` pide sólo la línea de estado y los headers de `/`, sin el body.

**Deberías ver** (varios valores cambian):

```text
HTTP/1.1 200 OK
Date: Tue, 20 Oct 2026 22:52:05 GMT
Content-Type: text/html; charset=utf-8
Connection: close
Server: cloudflare
Last-Modified: Tue, 13 Oct 2026 22:05:48 GMT
Allow: GET, HEAD
Accept-Ranges: bytes
Age: 5
cf-cache-status: HIT
```

- Un servidor que no conoces entendió tu texto escrito a mano: eso es un **protocolo**.
- `Server: cloudflare` y `cf-cache-status: HIT`: no contestó el servidor de example.com, sino una copia guardada en Cloudflare, una empresa que se pone en medio. La página 4 explica por qué eso es posible.

### La línea inicial

**En la petición** (la **línea de petición**) hay tres campos separados por espacios:

| Campo | Ejemplo | Qué dice |
|---|---|---|
| **Método** | `GET`, `POST`, `PUT`, `DELETE` | Qué quieres hacer. La página 3 dice qué promete cada uno |
| **Ruta** (*request-target*) | `/ventas/1?formato=csv` | Sobre qué recurso de ese servidor |
| **Versión** | `HTTP/1.1` | Qué versión del protocolo hablas |

**En la respuesta** (la **línea de estado**) también hay tres:

| Campo | Ejemplo | Qué dice |
|---|---|---|
| **Versión** | `HTTP/1.1` | Qué versión habla el servidor |
| **Código de estado** | `201` | Cómo le fue, en un número de tres dígitos. **Es lo que lee tu programa** |
| **Frase** | `Created` | El mismo resultado en palabras, para humanos. Los programas la ignoran |

### Headers: información sobre el mensaje

**Definición.** Un **header** es una línea `Nombre: valor` que da información sobre el mensaje, no los datos mismos. El nombre no distingue mayúsculas: `content-type` y `Content-Type` son el mismo header.

**Para qué existen.** Para que el receptor sepa **cómo leer** el body y **qué hacer** con el mensaje sin tener que adivinar. Agrupados por propósito:

| Propósito | Header | Ejemplo | Qué dice |
|---|---|---|---|
| **Destino** | `Host` | `api.github.com` | A qué sitio va. Una sola máquina sirve muchos sitios; con esto sabe cuál |
| **Describir el body** | `Content-Type` | `application/json` | En qué formato está el body |
| | `Content-Length` | `75` | Cuántos bytes mide el body: así el receptor sabe dónde termina el mensaje |
| **Negociar** | `Accept` | `text/csv` | En qué formatos aceptas la respuesta |
| **Identificarte** | `Authorization` | `Bearer fdd-2026` | Tu credencial: un **token**, una cadena secreta que prueba quién eres |
| | `User-Agent` | `python-requests/2.34.2` | Qué programa hace la petición |
| **Caché** | `ETag` / `If-None-Match` | `"d788c34b28aa"` | Una huella de la versión de los datos (página 4) |
| **Siguiente paso** | `Location` | `/ventas/5` | Dónde quedó lo que creaste |
| | `Allow` | `GET, HEAD` | Qué métodos acepta ese recurso |
| | `Retry-After` | `30` | Cuántos segundos esperar antes de reintentar |

**En la vida real** las APIs agregan headers propios:

- **GitHub** pide `Accept: application/vnd.github+json` (un formato JSON propio) y `X-GitHub-Api-Version: 2022-11-28` para fijar la versión del contrato; así un cambio futuro no rompe tu código.
- **Banxico** (su API de series económicas) recibe el token en un header propio, `Bmx-Token`.
- **GitHub** rechaza con `403` toda petición sin `User-Agent`. `requests` lo manda solo; un socket escrito a mano, no.

El prefijo `X-` en nombres de headers propios es una costumbre vieja; el RFC 6648 (2012) recomienda ya no usarlo, pero sigue en muchas APIs.

**Haz (celda 2.3):** una petición con `requests`, vista con `ver()`.

## 2.3 · Lo que manda requests

In [ ]:
r = requests.get(f"{URL}/ventas/1")
ver(r)

**Qué hace cada línea:**

- `requests.get(...)` — arma y manda la misma petición que escribiste a mano en la 2.1.
- `ver(r)` — imprime con `>>` lo que se mandó y con `<<` lo que llegó, en formato HTTP.

**Deberías ver:**

```text
>> GET /ventas/1 HTTP/1.1
>> User-Agent: python-requests/2.34.2
>> Accept-Encoding: gzip, deflate, zstd
>> Accept: */*
>> Connection: keep-alive
>>

<< HTTP/1.1 200 OK
<< Server: VentasFDD/1.0
<< Date: Tue, 20 Oct 2026 22:52:06 GMT
<< ETag: "d788c34b28aa"
<< Vary: Accept
<< Content-Type: application/json
<< Content-Length: 81
<<
<< {"id": 1, "tienda": "Centro", "producto": "café", "cantidad": 3, "precio": 45.0}
```

- **`requests` agregó cuatro headers sin que se los pidieras**: quién eres (`User-Agent`), qué compresión entiendes (`Accept-Encoding`), qué formatos aceptas (`*/*`: cualquiera) y que la conexión siga abierta (`keep-alive`).
- `ver()` no muestra `Host`: lo agrega la capa de abajo de `requests` justo antes de mandar. En la terminal del servidor sí aparece.

### Body: los datos

**Definición.** El **body** es la secuencia de bytes que va después de la línea en blanco. Es opcional: un `GET` normalmente no lleva; un `POST` que crea algo, sí.

Dos headers dicen cómo interpretarlo y dónde termina:

- **`Content-Type`** dice **cómo interpretar** los bytes (JSON, CSV, HTML, imagen).
- **`Content-Length`** dice **cuántos** bytes son, para que el receptor sepa dónde termina este mensaje y empieza el siguiente en la misma conexión.

**Haz (celda 2.4):** crea una venta mandando un body JSON.

## 2.4 · Un body JSON

In [ ]:
nueva = {"tienda": "Norte", "producto": "café", "cantidad": 2, "precio": 45.0}
r = requests.post(f"{URL}/ventas", json=nueva, headers=TOKEN)
ver(r)

**Qué hace cada línea:**

- `json=nueva` — `requests` **serializa** el dict a texto JSON, lo pone en el body y agrega `Content-Type: application/json` y `Content-Length`.
- `headers=TOKEN` — agrega `Authorization: Bearer fdd-2026`; este servidor sólo deja escribir a quien trae ese token.

**Deberías ver:**

```text
>> POST /ventas HTTP/1.1
>> User-Agent: python-requests/2.34.2
>> Accept-Encoding: gzip, deflate, zstd
>> Accept: */*
>> Connection: keep-alive
>> Authorization: Bearer fdd-2026
>> Content-Length: 75
>> Content-Type: application/json
>>
>> {"tienda": "Norte", "producto": "caf\u00e9", "cantidad": 2, "precio": 45.0}

<< HTTP/1.1 201 Created
<< Server: VentasFDD/1.0
<< Date: Tue, 20 Oct 2026 22:52:06 GMT
<< Location: /ventas/5
<< Content-Type: application/json
<< Content-Length: 80
<<
<< {"tienda": "Norte", "producto": "café", "cantidad": 2, "precio": 45.0, "id": 5}
```

- **`201 Created` y `Location: /ventas/5`**: el servidor creó la venta y te dice dónde quedó.
- `caf\u00e9` — en la petición, `requests` escribe la `é` como la secuencia JSON `\u00e9` (por eso el body mide 75 bytes). El servidor la convierte en el mismo texto y responde con la `é` literal: es la misma palabra.
- **Mira la terminal del servidor**: ahí está tu token, `Bearer fdd-2026`, en texto plano.

### HTTPS: HTTP cifrado

Todo lo que viste viajó **en claro**: cualquier equipo entre tu máquina y el servidor (el router del wifi, tu proveedor de internet) puede leerlo, token incluido. **HTTPS** es HTTP dentro de **TLS**, un protocolo que:

- **cifra** el mensaje, para que nadie en medio lo lea;
- **verifica** que el servidor es quien dice ser, con un certificado;
- **detecta** si alguien alteró el mensaje en el camino.

Los mensajes HTTP son idénticos; sólo viajan dentro de TLS, normalmente por el puerto 443. **Regla:** un token o un dato personal sólo viaja por `https://`. Este servidor usa `http://` porque nunca sale de tu máquina.

**Haz (celda 2.5):** pide la misma información en dos formatos.

## 2.5 · Un recurso, dos formatos

In [ ]:
for formato in ["application/json", "text/csv"]:
    r = requests.get(f"{URL}/ventas", params={"tienda": "Centro"},
                     headers={"Accept": formato})
    print(f"Accept: {formato}  →  Content-Type: {r.headers['Content-Type']}")
    print(r.text)
    print()

**Qué hace cada línea:**

- `params={"tienda": "Centro"}` — `requests` agrega `?tienda=Centro` a la URL: sólo las ventas del Centro.
- `headers={"Accept": formato}` — le dice al servidor en qué formato quieres la respuesta.
- `r.headers['Content-Type']` — en qué formato contestó de verdad.

**Deberías ver:**

```text
Accept: application/json  →  Content-Type: application/json
[{"id": 1, "tienda": "Centro", "producto": "café", "cantidad": 3, "precio": 45.0}]

Accept: text/csv  →  Content-Type: text/csv; charset=utf-8
id,tienda,producto,cantidad,precio
1,Centro,café,3,45.0
```

- **Misma URL, mismos datos, dos formatos.** El cliente pidió con `Accept`; el servidor avisó qué mandó con `Content-Type`. Esto se llama **negociación de contenido**.

**Haz (celda 2.6):** un endpoint viejo que omite un dato en `Content-Type`.

## 2.6 · El charset que faltaba

In [ ]:
r = requests.get(f"{URL}/exportar")
print("Content-Type:", r.headers["Content-Type"])
print("requests decodificó con:", r.encoding)
print(r.text.splitlines()[1])
r.encoding = "utf-8"
print(r.text.splitlines()[1])

**Qué hace cada línea:**

- `r.headers["Content-Type"]` — el formato que declaró el servidor.
- `r.encoding` — la **codificación** que `requests` usó para convertir los bytes del body en texto.
- `r.text.splitlines()[1]` — la primera fila de datos del CSV.
- `r.encoding = "utf-8"` — le decimos a `requests` la codificación correcta y volvemos a leer.

**Deberías ver:**

```text
Content-Type: text/csv
requests decodificó con: ISO-8859-1
1,Centro,cafÃ©,3,45.0
1,Centro,café,3,45.0
```

- **El body es bytes; para volverlo texto hay que saber la codificación.** La `é` en UTF-8 son dos bytes. Leídos con otra codificación (ISO-8859-1), salen dos caracteres: `Ã©`.
- El servidor mandó `text/csv` **sin** `charset=utf-8`. Sin ese dato, `requests` aplica la regla vieja de HTTP para texto: ISO-8859-1.
- **Si lo ignoras**: tu pipeline guarda `cafÃ©` sin ningún error, y nadie lo nota hasta que una cuenta por producto sale mal.

### La URL, parte por parte

**Definición.** Una **URL** (*Uniform Resource Locator*) dice dónde está un recurso y cómo llegar a él. El navegador, `requests` y `curl` la parten para armar la petición.

```text
  https://api.github.com:443/repos/python/cpython/issues?state=open&per_page=2#comentarios
  └─┬─┘   └─────┬──────┘ └┬┘└────────────┬────────────┘ └────────┬────────┘ └────┬────┘
 esquema       host    puerto          ruta (path)           query          fragmento
```

| Parte | Qué es | ¿Viaja al servidor? |
|---|---|---|
| **Esquema** | Qué protocolo usar: `http` o `https` | Decide cómo conectarse; no va en el mensaje |
| **Host** | El nombre del servidor; se traduce a una dirección IP | Sí, en el header `Host` |
| **Puerto** | En qué puerto escucha. Si se omite: 80 para `http`, 443 para `https` | Decide a dónde conectarse |
| **Ruta** | Qué recurso, dentro de ese servidor | Sí, en la línea de petición |
| **Query** | Pares `clave=valor` después de `?`, separados por `&` | Sí, en la línea de petición |
| **Fragmento** | Lo que va después de `#`: una posición dentro del documento | **No.** Lo usa sólo el navegador |

**La costumbre**: la ruta **identifica** el recurso (`/repos/python/cpython/issues`) y la query lo **filtra o pagina** (`state=open`, `per_page=2`). Es una convención de diseño, no una regla del protocolo.

**Haz (celda 2.7):** parte una URL real con la librería estándar.

## 2.7 · La URL, parte por parte

In [ ]:
from urllib.parse import parse_qs, urlsplit

url = "https://api.github.com:443/repos/python/cpython/issues?state=open&per_page=2#comentarios"
partes = urlsplit(url)
print("esquema:  ", partes.scheme)
print("host:     ", partes.hostname)
print("puerto:   ", partes.port)
print("ruta:     ", partes.path)
print("query:    ", parse_qs(partes.query))
print("fragmento:", partes.fragment)

**Qué hace cada línea:**

- `urlsplit(url)` — parte la URL en sus piezas.
- `parse_qs(partes.query)` — convierte `state=open&per_page=2` en un dict; cada valor es una lista porque una clave puede repetirse.

**Deberías ver:**

```text
esquema:   https
host:      api.github.com
puerto:    443
ruta:      /repos/python/cpython/issues
query:     {'state': ['open'], 'per_page': ['2']}
fragmento: comentarios
```

- Con `requests` no armes la query a mano: pásala en `params=`, como en la 2.5. `requests` se encarga de **codificar** los caracteres especiales (en la query, un espacio viaja como `+` o como `%20`).

# Métodos y códigos de estado

Meta: saber qué promete cada método, qué dice cada status y cuándo es seguro repetir una petición.

Meta: saber qué promete cada método, qué dice cada status y cuándo es seguro repetir una petición.

| Término | Qué es | Lo ves con |
|---|---|---|
| **método** | La primera palabra de la línea de petición: qué le pides al servidor que haga con el recurso | la celda 3.1 |
| **método seguro** | Un método con el que el cliente **no pide cambiar nada** en el servidor (sólo leer) | la celda 3.2 |
| **método idempotente** | Un método que, repetido N veces igual, deja el servidor **en el mismo estado** que una sola vez | la celda 3.1 |
| **cacheable** | Una respuesta que un programa intermedio puede guardar y reusar sin volver a preguntar | la página 4 |
| **status** | El número de tres dígitos de la línea de estado: cómo le fue a la petición | la celda 3.6 |
| **timeout** | El tiempo máximo que el cliente espera una respuesta antes de rendirse | la celda 3.3 |
| **reintento** | Mandar otra vez la misma petición porque la primera falló o no contestó | la celda 3.4 |
| **`Idempotency-Key`** | Un header con un identificador único por operación, para que el servidor reconozca un reintento | la celda 3.5 |

### En corto

- **El método es una promesa pública**: navegadores, cachés, crawlers y librerías de reintento la leen y actúan sin conocer tu aplicación.
- **Idempotente habla del estado del servidor, no de la respuesta**: un segundo `DELETE` responde 404 y sigue siendo idempotente.
- **Un timeout significa «no sé», no «no pasó»**: reintentar sólo es seguro si la operación es idempotente.

### Las tres propiedades de un método

HTTP (RFC 9110, 2022) no sólo da nombres a los métodos: les asigna **propiedades** que el resto del mundo puede leer sin preguntarte.

- **Seguro.** **Definición:** el cliente no solicita ningún cambio de estado en el servidor. El servidor puede escribir un log o contar visitas; lo que importa es que el cliente no pidió cambiar nada.
- **Idempotente.** **Definición:** el efecto **sobre el estado del servidor** de N peticiones idénticas es el mismo que el de una. La **respuesta** puede cambiar: borrar algo dos veces da 204 y luego 404, y el servidor queda igual en ambos casos (sin ese recurso).
- **Cacheable.** **Definición:** la respuesta se puede guardar y reusar para una petición igual, sin volver al servidor.

**Para qué existen.** Entre tu programa y el servidor hay programas que **no conocen tu aplicación**: el navegador, una caché, un proxy, un crawler, la librería que reintenta. Ninguno sabe qué hace `/ventas/2`, pero todos saben qué promete `GET`. Con eso deciden si pueden adelantarse, guardar o repetir.

| Método | Qué pide | Seguro | Idempotente | Cacheable |
|---|---|:---:|:---:|:---:|
| **GET** | Dame el recurso | sí | sí | sí |
| **HEAD** | Dame sólo los headers de lo que GET daría | sí | sí | sí |
| **OPTIONS** | ¿Qué puedo hacer con este recurso? | sí | sí | no |
| **POST** | Procesa estos datos según la lógica del recurso (crear, enviar, calcular) | no | no | en la práctica, no |
| **PUT** | Reemplaza el recurso completo con esto (o créalo en esa URL) | no | sí | no |
| **PATCH** | Cambia sólo estas partes del recurso | no | no garantizado | no |
| **DELETE** | Borra el recurso | no | sí | no |

- **Todo método seguro es idempotente**: si no cambias nada, repetirlo tampoco cambia nada.
- **POST no significa «crear»**: significa «procesa esto». Crear es su uso más común, no su definición.
- **PUT es idempotente porque reemplaza**: mandar el mismo reemplazo tres veces deja lo mismo. **POST no lo es**: tres POST pueden crear tres cosas.

**Haz (celda 3.1):** repite cada método tres veces y cuenta cuántas ventas quedan.

## 3.1 · Repetir cada método tres veces

In [ ]:
requests.post(f"{URL}/reiniciar")  # sólo para la demo: vuelve a las 4 ventas iniciales

nueva = {"tienda": "Sur", "producto": "té", "cantidad": 1, "precio": 30.0}
reemplazo = {"tienda": "Norte", "producto": "pan dulce", "cantidad": 10, "precio": 18.5}
pruebas = [
    ("GET",    lambda: requests.get(f"{URL}/ventas/2")),
    ("PUT",    lambda: requests.put(f"{URL}/ventas/2", json=reemplazo, headers=TOKEN)),
    ("POST",   lambda: requests.post(f"{URL}/ventas", json=nueva, headers=TOKEN)),
    ("DELETE", lambda: requests.delete(f"{URL}/ventas/2", headers=TOKEN)),
]
print(f"{'método':7} {'status de cada llamada':24} ventas al final")
for metodo, llamada in pruebas:
    statuses = [llamada().status_code for _ in range(3)]
    print(f"{metodo:7} {str(statuses):24} {total()}")

**Qué hace cada línea:**

- `requests.post(f"{URL}/reiniciar")` — regresa el servidor a sus 4 ventas, para que todos vean los mismos números.
- `pruebas = [...]` — una lista de pares (nombre, función); `lambda: …` guarda la llamada sin hacerla todavía.
- `[llamada().status_code for _ in range(3)]` — hace la misma petición tres veces y junta los tres status.
- `total()` — pregunta con un `GET` cuántas ventas hay ahora (viene de `cable.py`).

**Deberías ver:**

```text
método  status de cada llamada   ventas al final
GET     [200, 200, 200]          4
PUT     [200, 200, 200]          4
POST    [201, 201, 201]          7
DELETE  [204, 404, 404]          6
```

- **GET y PUT**: tres llamadas, el mismo estado que con una. **POST**: tres llamadas, tres ventas nuevas (de 4 a 7).
- **DELETE**: responde 204 y luego 404, **pero el servidor queda igual** después de la primera (sin la venta 2). Por eso es idempotente: la propiedad habla del estado, no de la respuesta.
- En la terminal del servidor ves las 12 peticiones, una por una.

### Seguro: lo que pasa cuando un GET cambia algo

**Si lo ignoras.** Un programa que sigue enlaces cree, con razón, que `GET` no cambia nada, y los sigue todos.

**En la vida real.**

- **2005, Google Web Accelerator.** Esta extensión descargaba por adelantado los enlaces de cada página, para que abrieran más rápido. Algunas aplicaciones web, entre ellas productos de 37signals, borraban datos con enlaces `GET`. El acelerador «visitó» esos enlaces y se perdieron datos de usuarios. Google respondió que esas aplicaciones no respetaban el estándar.
- **Hoy, escáneres de correo.** Los filtros de seguridad corporativos abren los enlaces de los correos para revisarlos. Un enlace de inicio de sesión de un solo uso que se consume con un `GET` queda gastado antes de que la persona lo abra. La solución que documentan Supabase y Stytch: el `GET` sólo muestra una página con un botón, y el botón manda un `POST`.

En el servidor de la clase, `GET /borrar?id=…` borra una venta. **Es mal diseño, a propósito**: sirve para ver el daño. `GET /panel` es una página HTML con un enlace a `/borrar` por cada venta.

**Haz (celda 3.2):** un crawler de cinco líneas que sigue todos los enlaces del panel.

## 3.2 · El crawler que borra

In [ ]:
requests.post(f"{URL}/reiniciar")  # sólo para la demo

print("antes:  ", total(), "ventas")
panel = requests.get(f"{URL}/panel").text
enlaces = re.findall(r'href="(.*?)"', panel)
for enlace in enlaces:
    requests.get(f"{URL}{enlace}")
    print("  seguí", enlace)
print("después:", total(), "ventas")

**Qué hace cada línea:**

- `requests.get(f"{URL}/panel").text` — descarga el HTML del panel como texto.
- `re.findall(r'href="(.*?)"', panel)` — saca con una expresión regular la dirección de cada enlace (`href="…"`); `(.*?)` captura lo que hay entre las comillas, lo menos posible.
- `requests.get(f"{URL}{enlace}")` — visita cada enlace con `GET`, que es lo que hace cualquier crawler o prefetcher.

**Deberías ver:**

```text
antes:   4 ventas
  seguí /borrar?id=1
  seguí /borrar?id=2
  seguí /borrar?id=3
  seguí /borrar?id=4
después: 0 ventas
```

- El crawler no hizo nada incorrecto: siguió enlaces `GET`, que prometen no cambiar nada. **El error es del servidor**, que rompió la promesa.
- **Regla de diseño:** lo que cambia estado va en `POST`, `PUT`, `PATCH` o `DELETE`. Nunca en `GET`.

### Timeout: cuando no llega respuesta

**Definición.** Un **timeout** es el límite de tiempo que el cliente espera; al pasarlo, deja de esperar y reporta un error.

**Para qué existe.** Sin límite, un servidor lento o caído deja a tu programa esperando para siempre. **`requests` no tiene timeout por defecto**: si no pasas `timeout=`, espera indefinidamente.

**Haz (celda 3.3):** pide algo a un servidor que tarda 3 segundos, con un límite de 1.

## 3.3 · Timeout

In [ ]:
inicio = time.perf_counter()
try:
    requests.get(f"{URL}/ventas", params={"lento": 3}, timeout=1)
except requests.Timeout as e:
    print("Timeout:", type(e).__name__)
print(f"esperé {time.perf_counter() - inicio:.1f} s")

**Qué hace cada línea:**

- `params={"lento": 3}` — agrega `?lento=3` a la URL; el servidor de la clase espera 3 segundos antes de contestar.
- `timeout=1` — límite para conectarse y para cada espera de datos durante la lectura; **no** es un límite de la duración total. Aquí vence porque en 1 segundo no llega ningún byte.
- `except requests.Timeout` — atrapa el error que `requests` lanza al vencer el límite.

**Deberías ver:**

```text
Timeout: ReadTimeout
esperé 1.0 s
```

- `ReadTimeout`: la conexión se abrió, pero la respuesta no llegó a tiempo. **El servidor sí recibió la petición** y siguió trabajando.

### Reintentar: el cobro doble

Un timeout sólo dice que **no sabes** qué pasó. Hay tres posibilidades y el cliente no puede distinguirlas:

```text
(a) cliente ── POST ──────X                        servidor   (nunca llegó)
(b) cliente ── POST ─────────────────────────────▶ servidor   (la guardó)
    cliente ◀──────X──────── 201 ──────────────── servidor   (la respuesta se perdió)
(c) cliente ── POST ─────────────────────────────▶ servidor   (sigue trabajando)

    En los tres casos el cliente sólo ve lo mismo: 1 s sin respuesta → timeout, «no sé».
```

Si reintentas y era (b) o (c), **la operación se hace dos veces**. Con un `GET` o un `PUT` no importa: son idempotentes. Con un `POST` que crea una venta, o que cobra una tarjeta, sí importa.

**Haz (celda 3.4):** un POST lento, con timeout, y un reintento.

## 3.4 · El cobro doble

In [ ]:
requests.post(f"{URL}/reiniciar")  # sólo para la demo

venta_nueva = {"tienda": "Centro", "producto": "café", "cantidad": 1, "precio": 45.0}
for intento in [1, 2]:
    try:
        requests.post(f"{URL}/ventas", params={"lento": 2}, json=venta_nueva,
                      headers=TOKEN, timeout=1)
        print(f"intento {intento}: llegó la respuesta")
    except requests.Timeout:
        print(f"intento {intento}: timeout, no sé si se guardó")
time.sleep(2.5)
print("ventas:", total(), "(había 4)")

**Qué hace cada línea:**

- `params={"lento": 2}` con `timeout=1` — el servidor tarda 2 segundos; el cliente se rinde al segundo.
- `for intento in [1, 2]` — el reintento ingenuo: si falló, mándalo otra vez.
- `time.sleep(2.5)` — espera a que el servidor, que sigue trabajando, termine las dos peticiones antes de contar.

**Deberías ver:**

```text
intento 1: timeout, no sé si se guardó
intento 2: timeout, no sé si se guardó
ventas: 6 (había 4)
```

- El cliente cree que **ninguna** se guardó; el servidor guardó **las dos**. En su terminal ves los dos `POST /ventas` completos y dos `respondí 201`.
- Ésta es la forma más común de duplicar un pedido o un cobro.

### La solución: `Idempotency-Key`

**Definición.** `Idempotency-Key` es un header con un identificador que **el cliente genera una vez por operación** y repite en cada reintento. El servidor guarda la primera respuesta para esa llave; si la llave vuelve, **devuelve la respuesta guardada sin repetir la operación**.

**Para qué existe.** Convierte un `POST` en algo que se puede reintentar sin daño.

**En la vida real.** Stripe, la plataforma de pagos, lo usa en sus `POST`, guarda la respuesta y marca las repetidas. Hubo un borrador del IETF para estandarizarlo, pero **expiró en 2026 sin convertirse en estándar**. Es una convención extendida, no una regla de HTTP.

**Haz (celda 3.5):** el mismo POST tres veces, con la misma llave.

## 3.5 · Idempotency-Key

In [ ]:
requests.post(f"{URL}/reiniciar")  # sólo para la demo

llave = {"Idempotency-Key": "pedido-7f3a", **TOKEN}
for intento in [1, 2, 3]:
    r = requests.post(f"{URL}/ventas", json=venta_nueva, headers=llave)
    print(f"intento {intento}: {r.status_code}  id={r.json()['id']}  "
          f"repetida={r.headers.get('Idempotent-Replayed', 'no')}")
print("ventas:", total(), "(había 4)")

**Qué hace cada línea:**

- `{"Idempotency-Key": "pedido-7f3a", **TOKEN}` — un diccionario de headers con la llave y el token; `**TOKEN` copia el header `Authorization`.
- `r.headers.get('Idempotent-Replayed', 'no')` — el servidor de la clase, igual que Stripe, marca con este header las respuestas que repitió de su memoria.

**Deberías ver:**

```text
intento 1: 201  id=5  repetida=no
intento 2: 201  id=5  repetida=true
intento 3: 201  id=5  repetida=true
ventas: 5 (había 4)
```

- Tres envíos, **una sola venta** (`id=5` las tres veces). Los intentos 2 y 3 no tocaron los datos.
- La llave la genera el cliente **antes** del primer intento; si la cambias en cada intento, no protege nada.

### Códigos de estado: quién tiene que actuar

**Definición.** El **status** es un número de tres dígitos en la línea de estado. El primer dígito es la **clase**; los otros dos, el caso.

**Para qué existe.** Un programa decide qué hacer **sin leer el body**. La clase da la primera orientación; el código exacto y sus headers deciden: corregir, autenticarse, esperar, resolver un conflicto o reintentar.

| Clase | Significa | Quién actúa |
|---|---|---|
| **1xx** | Información intermedia; la respuesta final viene después | nadie (casi nunca lo ves) |
| **2xx** | Éxito | nadie: usa la respuesta |
| **3xx** | El recurso está en otro lado, o no cambió | el cliente sigue `Location` o usa su copia |
| **4xx** | El servidor rechazó la petición por algo del cliente o del estado actual del recurso | **el cliente**: según el código, corrige, se autentica, espera (429) o resuelve el conflicto (409) |
| **5xx** | El servidor no pudo completar una petición válida | **el cliente** reintenta más tarde, **sólo** si la operación es idempotente o lleva una `Idempotency-Key` |

| Código | Nombre | Cuándo |
|---|---|---|
| **200** | OK | Funcionó; el body trae el resultado |
| **201** | Created | Se creó un recurso; `Location` dice su URL |
| **202** | Accepted | Recibido, se procesará después; todavía no está hecho |
| **204** | No Content | Funcionó y no hay body (típico de `DELETE`) |
| **304** | Not Modified | Tu copia sigue vigente; no se manda body (página 4) |
| **400** | Bad Request | La petición está mal formada (body que no es JSON) |
| **401** | Unauthorized | **No autenticado**: falta credencial o no es válida. Lleva `WWW-Authenticate` |
| **403** | Forbidden | El servidor entendió la petición y **se niega a atenderla**; casi siempre, por falta de permiso |
| **404** | Not Found | Ese recurso no existe |
| **405** | Method Not Allowed | El recurso existe, ese método no. Lleva `Allow` con los permitidos |
| **409** | Conflict | Choca con el estado actual del recurso |
| **429** | Too Many Requests | Demasiadas peticiones; `Retry-After` dice cuándo volver |
| **500** | Internal Server Error | Error no previsto en el servidor |
| **503** | Service Unavailable | El servidor no puede atender ahora (sobrecarga, mantenimiento) |

- **401 se llama «Unauthorized» pero significa no autenticado**: el servidor no sabe quién eres. **403** significa que el servidor entendió y se niega; en este servidor, porque tu token sólo puede leer.

**En la vida real.**

- **Kubernetes** responde **409** si intentas reemplazar un objeto que alguien cambió desde que lo leíste (lo detecta con su campo `resourceVersion`). El mecanismo estándar de HTTP para lo mismo es mandar `If-Match` con la versión que leíste y recibir **412 Precondition Failed** si cambió.
- **GitHub**, al pasarte del límite de peticiones, responde **403 o 429** según el caso: el mismo problema con dos códigos.
- **Banxico** (su API SIE) respondió **400** a un token inválido el 2026-10-20, no 401. Funciona, pero un cliente genérico no sabrá que el problema es la credencial.

**Haz (celda 3.6):** provoca seis status distintos en el servidor de la clase.

## 3.6 · Galería de códigos de estado

In [ ]:
casos = [
    ("GET /ventas/1",          lambda: requests.get(f"{URL}/ventas/1")),
    ("POST /ventas sin token", lambda: requests.post(f"{URL}/ventas", json={})),
    ("POST /ventas lector",    lambda: requests.post(f"{URL}/ventas", json={},
                                         headers={"Authorization": "Bearer lector"})),
    ("POST /ventas body malo", lambda: requests.post(f"{URL}/ventas", data="hola", headers=TOKEN)),
    ("GET /ventas/99",         lambda: requests.get(f"{URL}/ventas/99")),
    ("PATCH /ventas/1",        lambda: requests.patch(f"{URL}/ventas/1", json={})),
]
for nombre, llamada in casos:
    r = llamada()
    print(f"{nombre:25} {r.status_code} {r.reason:20} "
          f"{r.json()['error'] if r.status_code >= 400 else ''}")
r = requests.patch(f"{URL}/ventas/1", json={})
print("Allow:", r.headers["Allow"])

**Qué hace cada línea:**

- `"Bearer lector"` — un token que el servidor reconoce, pero sólo con permiso de lectura.
- `data="hola"` — manda texto plano como body donde el servidor espera JSON.
- `r.reason` — la frase de la línea de estado (`Not Found`), para humanos.
- `r.json()['error']` — el servidor de la clase explica cada error en un JSON con la llave `error`.

**Deberías ver:**

```text
GET /ventas/1             200 OK
POST /ventas sin token    401 Unauthorized         falta el token o no es válido
POST /ventas lector       403 Forbidden            tu token es válido, pero sólo puede leer
POST /ventas body malo    400 Bad Request          el body no es JSON válido
GET /ventas/99            404 Not Found            no hay venta 99
PATCH /ventas/1           405 Method Not Allowed   este servidor no implementa PATCH
Allow: GET, HEAD, PUT, DELETE
```

- 401 contra 403: misma petición, distinta credencial, distinto culpable.
- El 405 dice qué **sí** puedes hacer (`Allow`). Un buen error orienta al cliente.

### El status es una promesa, no una prueba

El servidor escribe el status; nada obliga a que sea verdad. **jsonplaceholder** es una API pública de pruebas que acepta escrituras y responde 201, pero no guarda nada (lo comprobamos el 2026-10-20).

**Haz (celda 3.7):** crea un recurso y búscalo.

## 3.7 · El 201 que miente

In [ ]:
r = requests.post("https://jsonplaceholder.typicode.com/posts", json={"title": "hola"}, timeout=5)
print("POST:", r.status_code, r.reason, "· Location:", r.headers.get("Location"))
r2 = requests.get(r.headers["Location"], timeout=5)
print("GET :", r2.status_code, r2.reason)

**Qué hace cada línea:**

- `requests.post(...)` — pide crear un recurso en una API real de internet.
- `r.headers["Location"]` — la URL donde, según el 201, quedó el recurso nuevo.
- `requests.get(...)` — va a buscarlo ahí.

**Deberías ver:**

```text
POST: 201 Created · Location: https://jsonplaceholder.typicode.com/posts/101
GET : 404 Not Found
```

- El servidor dijo «creado, está aquí» y ahí no hay nada. En un pipeline de datos, **verifica lo que importa** (vuelve a leerlo, cuenta filas) en vez de confiar sólo en el 2xx.

# HTTP, REST y CRUD

Meta: distinguir los tres conceptos, saber qué exige REST y qué cuesta cumplirlo, y decidir entre REST y RPC sobre HTTP.

Meta: distinguir los tres conceptos, saber qué exige REST y qué cuesta cumplirlo, y decidir entre REST y RPC sobre HTTP.

| Término | Qué es | Lo ves con |
|---|---|---|
| **recurso** | Cualquier cosa con nombre propio que una API expone: una venta, la lista de ventas, un usuario | la ruta /ventas/1 |
| **representación** | Los bytes que describen el estado de un recurso en un formato concreto: JSON, CSV, HTML | el header Content-Type |
| **URI** | El identificador de un recurso; una URL es una URI que además dice dónde y cómo pedirlo | la celda 2.7 |
| **estilo arquitectónico** | Un conjunto de reglas de diseño que, si las cumples, dan ciertas propiedades al sistema | la tesis de Fielding |
| **restricción** | Una de esas reglas: algo que el sistema **no** puede hacer, a cambio de una propiedad | la tabla de abajo |
| **REST** | *Representational State Transfer*: un estilo arquitectónico definido por seis restricciones | esta página |
| **sin estado** | Cada petición trae todo lo necesario para entenderla; el servidor no depende de lo que recuerde de una petición anterior. Sí puede guardar datos (ventas, usuarios, permisos) | la celda 4.2 |
| **caché** | Una copia guardada de una respuesta, para no volver a pedirla completa | la celda 4.1 |
| **hipermedia** | Respuestas que traen los enlaces a lo que puedes hacer después | el header Link de GitHub |
| **CRUD** | *Create, Read, Update, Delete*: las cuatro operaciones básicas sobre datos guardados | la celda 4.4 |
| **RPC** | *Remote Procedure Call*: llamar a una operación por su nombre en otra máquina | POST /reiniciar |

### En corto

- **HTTP, REST y CRUD son tres conceptos distintos**: cómo viaja el mensaje, cómo se organiza la API, qué haces con los datos.
- **REST es un estilo con seis restricciones**, cada una con un costo; casi ninguna API que se llama «REST» las cumple todas.
- **El mapeo CRUD → métodos HTTP es una convención**, no una equivalencia: hay operaciones que no son CRUD, y está bien.

### Tres conceptos, tres preguntas

La confusión clásica es decir «una API REST es una API que usa GET, POST, PUT y DELETE para hacer CRUD». Esa frase mezcla tres cosas distintas, que se combinan pero ninguna contiene a otra:

| Capa | Qué es | Pregunta que contesta | Lo define |
|---|---|---|---|
| **HTTP** | Un **protocolo**: el formato de los mensajes y sus reglas | ¿Cómo viaja una petición y su respuesta? | RFC 9110 y RFC 9112 |
| **REST** | Un **estilo arquitectónico**: reglas para organizar una API | ¿Cómo se diseña y se organiza la API? | La tesis de Roy Fielding (2000) |
| **CRUD** | Un **conjunto de operaciones** sobre datos guardados; viene de las bases de datos | ¿Qué le haces a los datos? | Nadie en particular: es vocabulario |

```text
┌─ HTTP ──────────────┐   ┌─ REST ──────────────┐   ┌─ CRUD ──────────────┐
│ cómo viaja cada     │   │ cómo se organiza    │   │ qué haces con los   │
│ mensaje             │   │ la API              │   │ datos               │
│                     │   │                     │   │                     │
│ línea inicial,      │   │ recursos con URI,   │   │ crear, leer,        │
│ headers, body,      │   │ representaciones,   │   │ actualizar, borrar  │
│ métodos, status     │   │ sin estado, caché…  │   │                     │
└─────────────────────┘   └─────────────────────┘   └─────────────────────┘
   un protocolo              un estilo de diseño       un vocabulario
```

- **Cada capa existe sin las otras.** Puedes usar HTTP sin REST (las APIs de LLM lo hacen), diseñar REST sobre otro protocolo (Fielding no lo ata a HTTP), y hacer CRUD en SQL sin red.
- **Lo que las une es una convención**: REST se suele implementar sobre HTTP, y CRUD se suele mapear a sus métodos.

### Recurso, representación y URI

**Recurso.** Es la unidad de información que la API nombra. La venta 1 es un recurso; la lista de ventas de la tienda Centro es otro. Un recurso es la **idea**, no los bytes.

**Representación.** Son los bytes que el servidor manda para describir el **estado actual** de un recurso, en un formato. La misma venta puede representarse como JSON o como CSV: el recurso es uno, las representaciones son dos.

**URI** (*Uniform Resource Identifier*). Es el identificador de un recurso. `/ventas/1` es una referencia **relativa**: junto con la base `http://127.0.0.1:8026` forma la URI completa. Una **URL** es una URI que además dice cómo llegar (scheme, host, puerto).

**Ya lo viste.** En la celda 2.5 de el-mensaje-http pediste `/ventas?tienda=Centro` dos veces, cambiando sólo el header `Accept`: una respuesta llegó en JSON y la otra en CSV. **Mismo recurso, dos representaciones.** El header `Content-Type` de cada respuesta dijo cuál era cuál.

- **En la práctica**, muchas APIs públicas no usan `Accept` para esto: prefieren un parámetro (`?format=csv`) o una extensión (`/ventas.csv`). `Accept` es el mecanismo de HTTP; la extensión es más fácil de escribir en un navegador.

### REST: seis restricciones, cada una con un precio

REST lo definió **Roy Fielding en su tesis doctoral (2000)**, a partir de lo que hacía funcionar a la Web. No es un estándar ni una librería: es una lista de **restricciones**. Si tu sistema las cumple, gana ciertas propiedades; cada una cuesta algo.

| Restricción | Qué exige | Qué ganas | Qué cuesta |
|---|---|---|---|
| **Cliente-servidor** | Separar quien pide de quien guarda los datos | Cada lado evoluciona por su cuenta | Toda interacción cruza la red |
| **Sin estado** | Cada petición trae todo lo necesario; el servidor no guarda una sesión del cliente entre peticiones (los datos, sí) | Cualquier servidor puede contestar; escalar es agregar servidores | Cada petición repite datos (el token, los filtros) |
| **Caché** | Cada respuesta dice si se puede guardar y por cuánto tiempo | Menos peticiones y menos bytes | Puedes leer una copia vieja |
| **Interfaz uniforme** | Todos los recursos se manejan igual: URI, representaciones, mensajes autodescriptivos, hipermedia | Herramientas genéricas (navegadores, cachés, proxies) entienden cualquier API | Menos eficiente que una interfaz hecha a la medida |
| **Sistema en capas** | El cliente no sabe si habla con el servidor final o con un intermediario | Puedes poner cachés, balanceadores y firewalls en medio | Cada capa agrega latencia |
| **Código bajo demanda** (opcional) | El servidor puede mandar código que el cliente ejecuta | El cliente se extiende sin reinstalarse | Menos visibilidad; por eso es opcional |

La **interfaz uniforme** es la restricción central, y tiene cuatro partes:

1. **Identificación de recursos**: cada recurso tiene su URI.
2. **Manipulación por representaciones**: cambias un recurso mandando una representación (el JSON de un PUT), no llamando a una función del servidor.
3. **Mensajes autodescriptivos**: cada mensaje dice cómo leerlo (`Content-Type`, método, status).
4. **Hipermedia**: las respuestas traen los enlaces a los siguientes pasos.

### Caché: no volver a bajar lo que ya tienes

El servidor manda con cada respuesta un **ETag**: una huella del contenido. La próxima vez, el cliente la manda en `If-None-Match`. Si el recurso no cambió, el servidor contesta **304 Not Modified sin body**: «lo que tienes sigue siendo válido».

**Haz (celda 4.1):** pide la misma venta dos veces a tu servidor, y lo mismo a PokeAPI, una API pública real.

## 4.1 · ETag y 304

In [ ]:
r1 = requests.get(f"{URL}/ventas/1")
r2 = requests.get(f"{URL}/ventas/1", headers={"If-None-Match": r1.headers["ETag"]})
print(f"1a vez: {r1.status_code} {r1.reason:13} {len(r1.content):>7} bytes  ETag {r1.headers['ETag']}")
print(f"2a vez: {r2.status_code} {r2.reason:13} {len(r2.content):>7} bytes")

p = "https://pokeapi.co/api/v2/pokemon/pikachu"
r1 = requests.get(p, timeout=10)
r2 = requests.get(p, headers={"If-None-Match": r1.headers["ETag"]}, timeout=10)
print(f"pokeapi 1a vez: {r1.status_code} {len(r1.content):>7,} bytes")
print(f"pokeapi 2a vez: {r2.status_code} {len(r2.content):>7,} bytes")
print("Cache-Control:", r1.headers["Cache-Control"])

**Qué hace cada línea:**

- `r1.headers["ETag"]` — la huella que el servidor mandó con la primera respuesta.
- `headers={"If-None-Match": ...}` — la segunda petición dice «ya tengo esta versión; mándamela sólo si cambió».
- `len(r.content)` — cuántos bytes trajo el body.
- `timeout=10` — PokeAPI está en internet: si en 10 segundos no contesta, la celda se rinde en vez de colgarse (la página 3 explica por qué `requests` no pone uno solo).
- `r1.headers["Cache-Control"]` — cuánto tiempo dice el servidor que puedes guardar la respuesta.

**Deberías ver:**

```text
1a vez: 200 OK                 81 bytes  ETag "d788c34b28aa"
2a vez: 304 Not Modified        0 bytes
pokeapi 1a vez: 200 309,106 bytes
pokeapi 2a vez: 304       0 bytes
Cache-Control: public, max-age=86400, s-maxage=86400
```

- **309,106 bytes contra 0**: la segunda vez sólo viajaron los headers. Para una fuente de datos que consultas cada hora, esa es la diferencia entre bajar todo y bajar nada.
- `max-age=86400` — PokeAPI dice que la respuesta vale 86,400 segundos (un día).
- **Cada representación tiene su propio ETag**: el JSON y el CSV de la misma venta tienen huellas distintas. Por eso tu servidor manda también `Vary: Accept`, que avisa a las cachés que la respuesta depende del header `Accept` y no deben mezclar una con otra.
- **En la vida real**: la documentación de GitHub dice que una petición condicional contestada con 304 **no cuenta** contra tu límite principal de peticiones, siempre que vaya autenticada. Caché bien usada es cuota ahorrada.

### Sin estado: cualquier servidor puede contestar

Una API grande no corre en un servidor: corre en muchos, detrás de un **balanceador de carga** (un programa que reparte las peticiones entre servidores). Si el servidor A recordara que ya te autenticaste, tu siguiente petición tendría que ir **forzosamente** a A. Eso se llama *sticky session*.

- **En la vida real**: la guía *The Twelve-Factor App* (Heroku, ~2011) dice que los procesos son *stateless and share-nothing* y que las sticky sessions *should never be used or relied upon*. El estado vive en una base de datos, no en la memoria del servidor.
- **Si lo ignoras**: el día que A se cae, todos sus clientes pierden su sesión; y no puedes agregar servidores sin romper a los que ya estaban.

**Haz (celda 4.2):** una petición con token y la misma sin token.

## 4.2 · Sin estado

In [ ]:
venta_nueva = {"tienda": "Centro", "producto": "café", "cantidad": 1, "precio": 45.0}

r = requests.post(f"{URL}/ventas", json=venta_nueva, headers=TOKEN)
print("con token:", r.status_code)
r = requests.post(f"{URL}/ventas", json=venta_nueva)
print("sin token:", r.status_code, "— el servidor no recuerda la petición anterior")

**Qué hace cada línea:**

- `venta_nueva = {...}` — la venta que vas a crear, como diccionario de Python.
- `requests.post(..., headers=TOKEN)` — crea la venta mandando el header `Authorization`.
- `requests.post(...)` sin `headers` — la misma petición, un instante después, sin token.

**Deberías ver:**

```text
con token: 201
sin token: 401 — el servidor no recuerda la petición anterior
```

- **El servidor no te «conoce» por haber entrado antes**: cada petición se juzga sola. Por eso el token viaja en **cada** petición.

### Sistema en capas: no sabes quién te contestó

En la celda 2.2 de el-mensaje-http, `example.com` respondió con `Server: cloudflare` y `cf-cache-status: HIT`. **No te contestó el servidor de example.com**: te contestó una copia en un servidor de Cloudflare, una capa intermedia. Tu cliente no cambió nada para que eso pasara. Eso es la restricción de capas funcionando.

### Hipermedia: seguir el enlace, no construirlo

Una API con hipermedia te dice **a dónde ir después**. GitHub lo hace al paginar: la respuesta trae un header `Link` con la URL de la página siguiente, marcada `rel="next"`. Tu código sigue esa URL en lugar de armarla.

**Haz (celda 4.3):** sólo el profesor: GitHub permite 60 peticiones por hora por IP y todo el salón sale por la misma IP.

## 4.3 · GitHub: el header Link (sólo el profesor)

In [ ]:
r = requests.get("https://api.github.com/repos/python/cpython/issues", params={"per_page": 2},
                 headers={"Accept": "application/vnd.github+json", "X-GitHub-Api-Version": "2022-11-28"}, timeout=10)
print(r.status_code)
print("Link:", r.headers.get("Link"))
print("te quedan:", r.headers.get("X-RateLimit-Remaining"), "de", r.headers.get("X-RateLimit-Limit"))

**Qué hace cada línea:**

- `params={"per_page": 2}` — pide los issues de dos en dos, para que haya página siguiente.
- `"Accept": "application/vnd.github+json"` — el tipo de representación propio de GitHub.
- `"X-GitHub-Api-Version": "2022-11-28"` — fija la versión del contrato de la API.
- `r.headers.get("Link")` — el enlace a la página siguiente.
- `X-RateLimit-Remaining` y `X-RateLimit-Limit` — cuántas peticiones te quedan esta hora y cuántas tienes.

**Deberías ver** (el cursor de la URL y el número que queda cambian):

```text
200
Link: <https://api.github.com/repositories/81598961/issues?per_page=2&after=Y3Vyc29yOnYyOpLPAAABoVt_7ajPAAAAAWKzxis%3D&page=2>; rel="next"
te quedan: 33 de 60
```

- **La URL de la página 2 trae un cursor (`after=...`) que no podrías inventar**: si construyeras la URL tú, tu código se rompería cuando GitHub cambie su forma de paginar. Siguiendo `rel="next"`, no.
- **Es hipermedia parcial**: GitHub enlaza la paginación, pero no todas las acciones posibles. Por eso es un ejemplo, no la regla.

### Qué tan REST es una API: el modelo de Richardson

Leonard Richardson propuso cuatro niveles (2008) para ubicar una API respecto a REST. Sirve para leer APIs reales sin pelearse por la etiqueta.

| Nivel | Qué hace la API | Ejemplo real |
|---|---|---|
| **0** | HTTP es sólo transporte: el nombre de la operación va en el body o en la ruta, casi siempre con POST | SOAP, JSON-RPC; Slack Web API, con una ruta por operación (/api/chat.postMessage) |
| **1** | Separa recursos con su URI, pero usa un solo método para todo | Poco común hoy; aparece en APIs internas viejas |
| **2** | Recursos + métodos HTTP con su significado + status correctos | GitHub, y la mayoría de las APIs «REST» |
| **3** | Además, hipermedia: las respuestas enlazan los siguientes pasos | Casi nadie completo; GitHub enlaza la paginación |

- **Casi ninguna «API REST» es REST en el sentido de Fielding**, porque casi ninguna llega al nivel 3. En la práctica, «API REST» quiere decir «API de nivel 2 sobre HTTP y JSON».
- **Las APIs de LLM** (`POST /v1/messages`) son **RPC sobre HTTP**: mandas una petición para que el servidor *calcule* una respuesta, no para crear un recurso que luego vas a leer. No es un defecto: es otro diseño.

### CRUD: cuatro operaciones y una convención

**CRUD** son las cuatro operaciones sobre datos guardados: **crear, leer, actualizar, borrar**. Vienen de las bases de datos (en SQL: INSERT, SELECT, UPDATE, DELETE). Al diseñar una API HTTP, se **acostumbra** mapearlas así:

| Operación CRUD | Método HTTP habitual | Status habitual | Nota |
|---|---|---|---|
| **Create** | POST a la colección | 201 Created + Location | PUT a una URI que el cliente ya conoce también crea (201) |
| **Read** | GET | 200 OK | Seguro: no cambia nada |
| **Update** | PUT (reemplaza completo) o PATCH (cambia una parte) | 200 OK o 204 | PUT es idempotente; PATCH no está obligado a serlo |
| **Delete** | DELETE | 204 No Content | La segunda vez: 404, y sigue siendo idempotente |

- **Es una convención, no una equivalencia.** POST no significa «crear»: significa «procesa esto según lo que defina el recurso». PUT puede crear. Y hay operaciones que no son ninguna de las cuatro.

**Haz (celda 4.4):** el ciclo completo sobre una venta, imprimiendo la tabla después de cada paso.

## 4.4 · Ciclo CRUD completo

In [ ]:
# Regresa los datos al inicio: las celdas anteriores crearon y borraron ventas.
requests.post(f"{URL}/reiniciar")

def tabla():
    for v in requests.get(f"{URL}/ventas").json():
        print(f"   {v['id']:>2} {v['tienda']:8} {v['producto']:10} {v['cantidad']:>3} x {v['precio']}")

print("Create · POST /ventas")
r = requests.post(f"{URL}/ventas", json={"tienda": "Sur", "producto": "café", "cantidad": 5, "precio": 45.0}, headers=TOKEN)
print("   ", r.status_code, "Location:", r.headers["Location"])
nueva = r.headers["Location"]
print("Read · GET", nueva)
print("   ", requests.get(f"{URL}{nueva}").json())
print("Update · PUT", nueva)
requests.put(f"{URL}{nueva}", json={"tienda": "Sur", "producto": "café", "cantidad": 50, "precio": 45.0}, headers=TOKEN)
tabla()
print("Delete · DELETE", nueva)
print("   ", requests.delete(f"{URL}{nueva}", headers=TOKEN).status_code)
tabla()

**Qué hace cada línea:**

- `requests.post(f"{URL}/reiniciar")` — vuelve a las cuatro ventas iniciales, para que tu salida coincida con la de abajo.
- `def tabla()` — pide todas las ventas con GET y las imprime en columnas.
- `r.headers["Location"]` — la URI del recurso recién creado; el servidor te dice dónde quedó.
- `requests.put(...)` — reemplaza la venta completa: manda **todos** los campos, no sólo la cantidad.
- `requests.delete(...)` — borra la venta; 204 dice «hecho, no hay nada que devolver».

**Deberías ver:**

```text
Create · POST /ventas
    201 Location: /ventas/5
Read · GET /ventas/5
    {'tienda': 'Sur', 'producto': 'café', 'cantidad': 5, 'precio': 45.0, 'id': 5}
Update · PUT /ventas/5
    1 Centro   café         3 x 45.0
    2 Norte    pan dulce    6 x 18.5
    3 Sur      té           2 x 30.0
    4 Polanco  café         1 x 52.0
    5 Sur      café        50 x 45.0
Delete · DELETE /ventas/5
    204
    1 Centro   café         3 x 45.0
    2 Norte    pan dulce    6 x 18.5
    3 Sur      té           2 x 30.0
    4 Polanco  café         1 x 52.0
```

- **El cliente nunca inventó la URI `/ventas/5`**: la leyó del header `Location`.
- **`/reiniciar` no es CRUD**: es una acción con un verbo en la ruta. Es RPC sobre HTTP, puesto a propósito en este servidor para que lo reconozcas.

### Acciones que no son CRUD

Cancelar un pedido, enviar un correo, aprobar un pago: son **acciones**, no «crear, leer, actualizar, borrar». Hay dos salidas comunes:

- **Método personalizado.** Google (guía de diseño AIP-136) agrega la acción después de dos puntos: `POST /v1/pedidos/7:cancel`. Es RPC declarado, con POST porque cambia estado.
- **Convertir la acción en recurso.** `POST /pedidos/7/cancelaciones` crea una cancelación. Es nivel 2 puro, a costa de inventar un sustantivo.

### Decidir: ¿REST o RPC sobre HTTP?

No hay una respuesta correcta para todo. Cada diseño tiene ventajas y costos distintos:

| Si tu API… | Inclínate por | Por qué |
|---|---|---|
| Expone **entidades** que se crean, leen y cambian (ventas, usuarios, archivos) | **Orientada a recursos (nivel 2)** | Los métodos con su significado te dejan aprovechar herramientas genéricas; tú implementas sus garantías, defines la caché y decides qué se puede reintentar |
| Expone **acciones o cálculos** (generar texto, convertir un archivo, enviar un mensaje) | **RPC sobre HTTP** | No hay recurso que leer después; forzar un sustantivo complica sin ganar nada |
| Tiene **muchos consumidores que no conoces** | **Orientada a recursos** | La interfaz uniforme es la que todos ya saben leer |
| Tiene **pocos consumidores, conocidos** (tu equipo) | **Cualquiera**; RPC es más directo | El acuerdo cabe en una conversación |
| Necesita **caché** de lecturas | **GET sobre recursos** | Las cachés intermedias sólo guardan GET (y HEAD) sin configuración especial |
| Tiene **operaciones largas** (minutos) | **Un recurso para la operación** | Creas el trabajo, y su estado se consulta como cualquier otro recurso |

- **El costo de REST** es diseñar sustantivos y respetar la semántica de cada método.
- **El costo de RPC** es que los intermediarios no entienden tus operaciones: no pueden cachearlas ni saber si es seguro repetirlas. Todo eso lo tienes que resolver tú.

### Lo que esta clase no cubre

Para que sepas qué contestar si sale la pregunta: existe, y no es de esta clase.

- **Polling y webhooks** — cómo te enteras de que algo terminó: preguntando cada cierto tiempo, o dejando que el servidor te llame.
- **Streaming** — recibir una respuesta por partes mientras se genera, en lugar de esperar a que esté completa.
- **GraphQL** — un estilo donde el cliente escribe una consulta con los campos exactos que quiere.
- **gRPC** — RPC con mensajes binarios y un contrato escrito en un archivo `.proto`, común entre servicios internos.
- **OAuth** — un protocolo para que una aplicación actúe en tu nombre sin conocer tu contraseña.
- **Paginación a fondo** — partir respuestas grandes en páginas: por número de página o por cursor.
- **Versionado** — cómo cambia una API sin romper a quien ya la usa.